In [ ]:
import gradio as gr
import google.generativeai as genai  # Using Gemini for both steps
import numpy as np
from pymongo import MongoClient
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Gemini API
genai.configure(api_key="") #useUrOwnApiKey

# Connect to MongoDB
client = MongoClient("") #useUrOwnMongoDB
db = client["tenders_db"]
collection = db["tenders"]

# Load the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    """Generates an embedding using a local model."""
    return embedding_model.encode(text, convert_to_numpy=True).tolist()  # Convert to list for MongoDB compatibility

#Load tender data from CSV
file_path = r"C:\Projects\Bachlorz\cleaned_lemmatized_data.csv"  # Update with the correct path
df = pd.read_csv(file_path, encoding="utf-8")

# Combine all text columns into a single field
df["combined_text"] = df.astype(str).apply(lambda x: " ".join(x), axis=1)

# Insert data into MongoDB
for _, row in df.iterrows():
    tender_doc = {
        "title": row["Tender Title"],
        "text": row["combined_text"],
        "embedding": get_embedding(row["combined_text"]),  # Store embedding
    }
    collection.insert_one(tender_doc)

print("✅ Data inserted into MongoDB successfully!")

✅ Data inserted into MongoDB successfully!


In [ ]:
def find_top_tenders_rag(query, top_n=5):
    """Retrieve top tenders from MongoDB and fix lemmatized text."""
    query_embedding = get_embedding(query)

    # Retrieve all tenders from MongoDB
    tenders = list(collection.find({}, {"title": 1, "text": 1, "embedding": 1}))

    # Compute similarity scores
    similarities = [
        (tender, cosine_similarity([query_embedding], [np.array(tender["embedding"])])[0][0])
        for tender in tenders
    ]

    # Sort by highest similarity and get top N tenders
    top_tenders = sorted(similarities, key=lambda x: x[1], reverse=True)[:top_n]

    # ✅ Format output with lemmatized text (to be cleaned later)
    tender_results = {}
    for i, tender in enumerate(top_tenders):
        tender_results[str(i)] = {
            "Tender Title": tender[0]["title"],
            "combined_text": tender[0]["text"][:1000]  # Limit text size
        }

    return tender_results


def fix_lemmatized_title_and_text_with_gemini(title, tender_text):
    """
    Uses Gemini to convert both the tender title and text into human-readable English.
    """
    prompt = f"""
    The following tender **title and text** are in **lemmatized format**, meaning words are in their base form, and the sentence structure is incomplete.

    **Your Task:**
    - Convert the **title** into a clear and natural English phrase.
    - Convert the **tender text** into **proper English sentences**.
    - Provide a **concise summary** (max 2 sentences).
    - Keep everything **professional and concise**.

    **Lemmatized Tender Title:**  
    {title}

    **Lemmatized Tender Text:**  
    {tender_text}

    **Now return the improved version in this format:**
    ---
    **Title:** [Fixed tender title]  
    **Human-Readable Version:** [Fixed tender text]  
    **Summary:** [2-sentence summary]  

    **Now generate the improved version:**
    """

    model = genai.GenerativeModel("gemini-pro")
    response = model.generate_content(prompt)

    return response.text.strip()


def generate_rag_response(query):
    """Retrieve tenders from MongoDB, fix their titles & text with Gemini, and generate a final response with Gemini."""

    retrieved_tenders = find_top_tenders_rag(query)

    if not retrieved_tenders:
        return "No relevant tenders found for this query."

    cleaned_tenders = {}
    for key, tender in retrieved_tenders.items():
        fixed_result = fix_lemmatized_title_and_text_with_gemini(
            tender["Tender Title"], tender["combined_text"]
        )
        cleaned_tenders[key] = {"Fixed Output": fixed_result}

    #formatting cleaned tenders 
    formatted_tenders = "\n\n".join([
        f"{t['Fixed Output']}" for t in cleaned_tenders.values()
    ])

    #Using Gemini again to generate a final answer
    prompt = f"""
    You are an AI assistant specializing in tenders and procurement.

    Below are the relevant tenders retrieved from the MongoDB database. These tenders were originally in **lemmatized format**, but have been reconstructed into natural language.

    **Your Task:**
    - Summarize the key information in clear, natural English.
    - Answer the query based on the retrieved tenders.
    - If the retrieved tenders do not contain relevant information, say so.

    **Retrieved Tenders:**
    {formatted_tenders}

    **Query:** "{query}"

    Provide a clear, structured answer:
    """

    model = genai.GenerativeModel("gemini-pro")
    response = model.generate_content(prompt)

    return cleaned_tenders, response.text.strip()

In [ ]:
# Gradio Interface
interface = gr.Interface(
    fn=generate_rag_response,
    inputs="text",
    outputs=[
        gr.JSON(label="Fixed Retrieved Tenders (Readable)"), 
        gr.Markdown(label="AI Generated Response")  
    ]
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7869

To create a public link, set `share=True` in `launch()`.
